# Laboratory 7 — One LIF Neuron in Nengo

Companion to Chapter 9. Use the repository environment (Nengo 4.0.0 and NumPy 1.26.4). In Colab, run `%pip install numpy==1.26.4 nengo==4.0.0 contourpy==1.3.3` before imports and restart the runtime if NumPy was already imported. Nengo current and voltage here are normalized values, not pA or mV.

Run all cells, then change one parameter at a time. Figures are synthetic teaching outputs.

[Run in Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/nengo_single_neuron.ipynb)

## One neuron, a direct current connection, and two separate output plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nengo
# EDIT normalized_current (threshold is 1), tau_rc, or tau_ref.
normalized_current, tau_rc, tau_ref = 1.5, .02, .002
with nengo.Network(seed=7) as network:
    stimulus = nengo.Node(lambda t: normalized_current if t >= .2 else 0.)
    cell = nengo.Ensemble(1,1,neuron_type=nengo.LIF(tau_rc=tau_rc,tau_ref=tau_ref),
                          gain=[2.],bias=[0.],encoders=[[1.]])
    # Nengo multiplies neuron input by gain=2; compensate to obtain J.
    nengo.Connection(stimulus,cell.neurons,transform=.5,synapse=None)
    events_probe = nengo.Probe(cell.neurons)
    voltage_probe = nengo.Probe(cell.neurons,'voltage')
with nengo.Simulator(network,dt=.001,progress_bar=False) as sim:
    sim.run(.8)
time = sim.trange()
event_times = time[sim.data[events_probe][:,0]>0]
fig, ax = plt.subplots(2,1,figsize=(10,5),sharex=True,constrained_layout=True,
                       gridspec_kw={'height_ratios':[3,1]})
ax[0].plot(time,sim.data[voltage_probe][:,0],color='#24628a'); ax[0].set(ylabel='Normalized voltage',title='Nengo: direct current to one LIF neuron')
ax[1].vlines(event_times,0,1,color='#b75d45'); ax[1].set(xlabel='Time (s)',ylabel='Events',yticks=[])
plt.show()
assert np.all(event_times>=.2)
if normalized_current <= 1:
    assert len(event_times) == 0
    print('At/below threshold: no output events in this deterministic example.')
elif len(event_times) >= 2:
    expected = 1/(tau_ref + tau_rc*np.log(normalized_current/(normalized_current-1)))
    measured = 1/np.diff(event_times).mean()
    assert abs(expected-measured)/expected < .06
    print(f'Nengo {nengo.__version__}: analytical {expected:.2f} Hz, simulated {measured:.2f} Hz')
else:
    print('Too few events to estimate an interval rate; extend the simulation.')

## References

[1] Nengo developers, “Frontend API,” ver. 4.0.0. [LIF, Ensemble, Connection, and Probe](https://www.nengo.ai/nengo/v4.0.0/frontend-api.html).

**Model scope.** This is a single-neuron API check, not a learned robot or a Nengo Oja implementation.